# 01. Exploratory Data Analysis (EDA) - Default of Credit Card Clients

**Đề tài**: Tối ưu đa mục tiêu siêu tham số XGBoost cho dự đoán rủi ro tín dụng cá nhân xét đến hiệu năng dự đoán và độ ổn định của giải thích SHAP  
**Bộ dữ liệu**: UCI Default of Credit Card Clients (Yeh & Lien, 2009)  
**Mục tiêu notebook**:
1. Khảo sát cấu trúc dữ liệu, kiểu dữ liệu và kiểm tra giá trị thiếu (missing values).
2. Phân tích phân bố nhãn mục tiêu (mất cân bằng lớp `default.payment.next.month`).
3. Kiểm tra các giá trị bất thường/không tài liệu hóa trong các biến phân loại (`EDUCATION`, `MARRIAGE`).
4. Khảo sát mối liên hệ giữa các biến lịch sử thanh toán trễ (`PAY_0` đến `PAY_6`) và tỷ lệ vỡ nợ.
5. Phân tích tương quan giữa hạn mức tín dụng (`LIMIT_BAL`), hóa đơn (`BILL_AMT`) và tiền trả (`PAY_AMT`).

In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Thêm src vào sys.path để tái sử dụng module dự án
sys.path.insert(0, str(Path.cwd().parent / "src"))
from creditrisk.config import load_data_config
from creditrisk.data import load_raw_data, standardize_columns

sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["figure.figsize"] = (10, 6)

## 1. Tải và Khảo sát Tổng quan Dữ liệu

In [ ]:
config = load_data_config("../configs/data.yaml")
df_raw = load_raw_data("../" + config.raw_data_path, config=config)
df = standardize_columns(df_raw, config=config)

print(f"Số dòng (observations): {df.shape[0]:,}")
print(f"Số cột (features + target): {df.shape[1]}")
df.head()

## 2. Kiểm tra Giá trị Thiếu (Missing Values) & Kiểu dữ liệu

In [ ]:
null_summary = df.isnull().sum()
print("Tổng số ô missing:", null_summary.sum())
df.info()

## 3. Phân bố Nhãn Mục Tiêu (Target Distribution)

In [ ]:
target_col = config.columns.target_column_clean
counts = df[target_col].value_counts()
percents = df[target_col].value_counts(normalize=True) * 100

summary_target = pd.DataFrame({"Số lượng": counts, "Tỷ lệ (%)": percents.round(2)})
display(summary_target)

fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(x=counts.index, y=counts.values, palette="Blues_r", ax=ax)
ax.set_title("Phân bố Nhãn Vỡ Nợ (default_payment_next_month)")
ax.set_xticklabels(["0 (Không vỡ nợ)", "1 (Vỡ nợ)"])
ax.set_ylabel("Số khách hàng")
plt.tight_layout()
plt.show()

## 4. Khảo sát Các Biến Phân loại (Demographics) & Các Giá trị Ngoại lệ
- `EDUCATION`: Các giá trị 0, 5, 6 là không rõ nghĩa hoặc undocumented.
- `MARRIAGE`: Giá trị 0 là undocumented.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
sns.countplot(data=df, x="SEX", hue=target_col, ax=axes[0])
axes[0].set_title("Giới tính vs Default")
axes[0].set_xticklabels(["Nam (1)", "Nữ (2)"])

sns.countplot(data=df, x="EDUCATION", hue=target_col, ax=axes[1])
axes[1].set_title("Học vấn vs Default")

sns.countplot(data=df, x="MARRIAGE", hue=target_col, ax=axes[2])
axes[2].set_title("Tình trạng Hôn nhân vs Default")
plt.tight_layout()
plt.show()

## 5. Mối Liên hệ Giữa Lịch Sử Chậm Trả (PAY_0 đến PAY_6) và Khả Năng Vỡ Nợ
Trạng thái chậm trả (>= 2 tháng) thường là tín hiệu rủi ro mạnh nhất.

In [ ]:
pay_cols = config.columns.payment_status_features
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for idx, col in enumerate(pay_cols):
    ax = axes[idx // 3, idx % 3]
    sns.countplot(data=df, x=col, hue=target_col, ax=ax)
    ax.set_title(f"Trạng thái thanh toán: {col}")
    ax.legend(title="Default", loc="upper right")
plt.tight_layout()
plt.show()

## 6. Phân tích Tương quan Đặc trưng Số (Numerical Correlation Matrix)

In [ ]:
num_cols = config.columns.numerical_features + config.columns.bill_amount_features + config.columns.payment_amount_features
corr = df[num_cols + [target_col]].corr(method="spearman")

plt.figure(figsize=(14, 12))
sns.heatmap(corr, cmap="coolwarm", annot=False, fmt=".2f", cbar=True)
plt.title("Spearman Correlation Heatmap (Continuous Features)")
plt.tight_layout()
plt.show()

## 7. Kết luận rút ra từ EDA cho Pipeline Tiền xử lý và Mô hình hóa
1. **Không có missing value thực thể**, nhưng có mã giá trị lạ trong `EDUCATION` (0, 5, 6) và `MARRIAGE` (0) -> Cần `CleanCategoricalTransformer` gom vào nhóm 4 và nhóm 3.
2. **Mất cân bằng lớp ~22.1%** -> Metric tối ưu chính phải là **ROC-AUC** (hoặc PR-AUC), không dùng Accuracy thông thường.
3. **Biến `PAY_0` đến `PAY_6` có tương quan rất cao với rủi ro vỡ nợ**, đặc biệt khi khách hàng bắt đầu trễ hạn từ 2 tháng trở lên.
4. **Dư nợ `BILL_AMT1..6` có tương quan đa cộng tuyến cao** giữa các tháng liền kề -> Các thuật toán cây (XGBoost, LightGBM, CatBoost) sẽ tự nhiên xử lý tốt hiện tượng này.